In [1]:
!pip install pandas
!pip install numpy
!pip install matplotlib
!pip install seaborn
!pip install beautifulsoup4
!pip install requests
!pip install sqlalchemy
!pip install pymysql
!pip install mysql-connector-python
!pip install faker

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
    --------------------------------------- 0.3/17.7 MB ? eta -:--:--
   - -------------------------------------- 0.5/17.7 MB 1.9 MB/s eta 0:00:09
   -- ------------------------------------- 1.0/17.7 MB 2.4 MB/s eta 0:00:08
   -

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import random
import re
import os
from datetime import datetime, timedelta

# Create data folder if not exists
os.makedirs("data", exist_ok=True)

print(" All libraries imported successfully")

 All libraries imported successfully


In [2]:
# SOURCE 1: books.toscrape.com — Books catalog
base_url = "http://books.toscrape.com/catalogue/page-{}.html"
books_products = []

rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

for page in range(1, 11):  # 10 pages = ~200 books
    url = base_url.format(page)
    response = requests.get(url)
    response.encoding = "utf-8"   # 🔑 FIX for the £ encoding error
    soup = BeautifulSoup(response.text, "html.parser")
    
    items = soup.find_all("article", class_="product_pod")
    for item in items:
        title = item.h3.a["title"]
        
        # 🔑 FIX: Use regex to extract numeric value only
        price_text = item.find("p", class_="price_color").text
        price_match = re.search(r"[\d.]+", price_text)
        price = float(price_match.group()) if price_match else 0.0
        
        rating_class = item.find("p", class_="star-rating")["class"][1]
        rating = rating_map.get(rating_class, 3)
        
        availability = item.find("p", class_="instock availability").text.strip()
        
        books_products.append({
            "product_name": title[:100],
            "category": "Books",
            "unit_price": round(price * 100, 2),   # convert £ → ₹-style
            "rating": rating,
            "availability": availability,
            "source": "books.toscrape.com"
        })

books_df = pd.DataFrame(books_products)
print(f" Source 1: Scraped {len(books_df)} books")
books_df.head()

 Source 1: Scraped 200 books


,product_name,category,unit_price,rating,availability,source
0,A Light in the Attic,Books,5177.0,3,In stock,books.toscrape.com
1,Tipping the Velvet,Books,5374.0,1,In stock,books.toscrape.com
2,Soumission,Books,5010.0,1,In stock,books.toscrape.com
3,Sharp Objects,Books,4782.0,4,In stock,books.toscrape.com
4,Sapiens: A Brief History of Humankind,Books,5423.0,5,In stock,books.toscrape.com


In [3]:
# SOURCE 2: webscraper.io test site — Computers / Electronics
electronics_products = []

ecom_urls = [
    "https://webscraper.io/test-sites/e-commerce/allinone/computers/laptops",
    "https://webscraper.io/test-sites/e-commerce/allinone/computers/tablets",
    "https://webscraper.io/test-sites/e-commerce/allinone/phones/touch"
]

category_map = {
    "laptops": "Electronics-Laptop",
    "tablets": "Electronics-Tablet",
    "touch":   "Electronics-Phone"
}

for url in ecom_urls:
    response = requests.get(url)
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")
    
    cat_key = url.split("/")[-1]
    category = category_map.get(cat_key, "Electronics")
    
    items = soup.find_all("div", class_="thumbnail")
    for item in items:
        title_tag = item.find("a", class_="title")
        price_tag = item.find("h4", class_="price")
        
        if title_tag and price_tag:
            title = title_tag.get("title", title_tag.text.strip())
            price_match = re.search(r"[\d.]+", price_tag.text)
            price = float(price_match.group()) if price_match else 0.0
            
            electronics_products.append({
                "product_name": title[:100],
                "category": category,
                "unit_price": round(price * 80, 2),   # USD → INR approx
                "rating": random.randint(3, 5),
                "availability": "In stock",
                "source": "webscraper.io"
            })

electronics_df = pd.DataFrame(electronics_products)
print(f" Source 2: Scraped {len(electronics_df)} electronics items")
electronics_df.head()

 Source 2: Scraped 147 electronics items


,product_name,category,unit_price,rating,availability,source
0,Asus VivoBook X441NA-GA190,Electronics-Laptop,23679.2,5,In stock,webscraper.io
1,Prestigio SmartBook 133S Dark Grey,Electronics-Laptop,23920.0,5,In stock,webscraper.io
2,Prestigio SmartBook 133S Gold,Electronics-Laptop,23920.0,4,In stock,webscraper.io
3,Aspire E1-510,Electronics-Laptop,24559.2,3,In stock,webscraper.io
4,Lenovo V110-15IAP,Electronics-Laptop,25755.2,3,In stock,webscraper.io


In [4]:
# SOURCE 3: quotes.toscrape.com — Use authors/tags to create "Merchandise" products
merchandise = []
quote_url = "http://quotes.toscrape.com/page/{}/"

for page in range(1, 6):
    response = requests.get(quote_url.format(page))
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")
    
    quotes = soup.find_all("div", class_="quote")
    for q in quotes:
        author = q.find("small", class_="author").text.strip()
        tags = [t.text for t in q.find_all("a", class_="tag")]
        main_tag = tags[0] if tags else "general"
        
        # Generate merchandise products
        for prod_type in ["T-Shirt", "Mug", "Poster"]:
            merchandise.append({
                "product_name": f"{author} {prod_type}"[:100],
                "category": f"Merchandise-{prod_type}",
                "unit_price": round(random.uniform(200, 1500), 2),
                "rating": random.randint(3, 5),
                "availability": "In stock",
                "source": "quotes.toscrape.com"
            })

merch_df = pd.DataFrame(merchandise).drop_duplicates(subset=["product_name"])
print(f" Source 3: Generated {len(merch_df)} merchandise items")
merch_df.head()

 Source 3: Generated 84 merchandise items


,product_name,category,unit_price,rating,availability,source
0,Albert Einstein T-Shirt,Merchandise-T-Shirt,623.68,4,In stock,quotes.toscrape.com
1,Albert Einstein Mug,Merchandise-Mug,217.14,5,In stock,quotes.toscrape.com
2,Albert Einstein Poster,Merchandise-Poster,1259.05,3,In stock,quotes.toscrape.com
3,J.K. Rowling T-Shirt,Merchandise-T-Shirt,806.78,4,In stock,quotes.toscrape.com
4,J.K. Rowling Mug,Merchandise-Mug,854.88,4,In stock,quotes.toscrape.com


In [5]:
# Combine all 3 sources into one master product catalog
products_df = pd.concat([books_df, electronics_df, merch_df], ignore_index=True)
products_df = products_df.drop_duplicates(subset=["product_name"]).reset_index(drop=True)
products_df["product_id"] = ["PROD" + str(1000 + i) for i in range(len(products_df))]

print(f" Total unique products across sources: {len(products_df)}")
print("\nProducts per source:")
print(products_df["source"].value_counts())
print("\nProducts per category:")
print(products_df["category"].value_counts())

products_df.to_csv("data/products_master.csv", index=False)
products_df.head(10)

 Total unique products across sources: 398

Products per source:
source
books.toscrape.com     200
webscraper.io          114
quotes.toscrape.com     84
Name: count, dtype: int64

Products per category:
category
Books                  200
Electronics-Laptop      88
Merchandise-T-Shirt     28
Merchandise-Mug         28
Merchandise-Poster      28
Electronics-Tablet      19
Electronics-Phone        7
Name: count, dtype: int64


,product_name,category,unit_price,rating,availability,source,product_id
0,A Light in the Attic,Books,5177.0,3,In stock,books.toscrape.com,PROD1000
1,Tipping the Velvet,Books,5374.0,1,In stock,books.toscrape.com,PROD1001
2,Soumission,Books,5010.0,1,In stock,books.toscrape.com,PROD1002
3,Sharp Objects,Books,4782.0,4,In stock,books.toscrape.com,PROD1003
4,Sapiens: A Brief History of Humankind,Books,5423.0,5,In stock,books.toscrape.com,PROD1004
5,The Requiem Red,Books,2265.0,1,In stock,books.toscrape.com,PROD1005
6,The Dirty Little Secrets of Getting Your Dream...,Books,3334.0,4,In stock,books.toscrape.com,PROD1006
7,The Coming Woman: A Novel Based on the Life of...,Books,1793.0,3,In stock,books.toscrape.com,PROD1007
8,The Boys in the Boat: Nine Americans and Their...,Books,2260.0,4,In stock,books.toscrape.com,PROD1008
9,The Black Maria,Books,5215.0,1,In stock,books.toscrape.com,PROD1009


In [6]:
# Master data lists (replacing Faker)
stores = ["Mumbai-01", "Delhi-02", "Bangalore-03", "Chennai-04", "Hyderabad-05"]
regions = {
    "Mumbai-01":   "West",
    "Delhi-02":    "North",
    "Bangalore-03":"South",
    "Chennai-04":  "South",
    "Hyderabad-05":"South"
}
payment_methods = ["Cash", "Card", "UPI", "Wallet", "Net Banking"]

# Generate customer IDs pool
customer_ids = ["CUST" + str(2000 + i) for i in range(2000)]

# Date range
start_date = datetime(2024, 1, 1)
end_date   = datetime(2024, 12, 31)
date_diff  = (end_date - start_date).days

sales_data = []
n_transactions = 10000

for i in range(n_transactions):
    product = products_df.sample(1).iloc[0]
    store = random.choice(stores)
    sale_date = start_date + timedelta(days=random.randint(0, date_diff))
    qty = random.randint(1, 5)
    discount = random.choice([0, 0.05, 0.10, 0.15, 0.20])
    unit_price = float(product["unit_price"])
    revenue = round(qty * unit_price * (1 - discount), 2)
    
    sales_data.append({
        "transaction_id": f"TXN{100000 + i}",
        "sale_date":      sale_date.strftime("%Y-%m-%d"),
        "store_id":       store,
        "region":         regions[store],
        "product_id":     product["product_id"],
        "product_name":   product["product_name"],
        "category":       product["category"],
        "quantity":       qty,
        "unit_price":     unit_price,
        "discount":       discount,
        "revenue":        revenue,
        "customer_id":    random.choice(customer_ids),
        "payment_method": random.choice(payment_methods)
    })

sales_df = pd.DataFrame(sales_data)
sales_df.to_csv("data/scraped_sales.csv", index=False)

print(f" Generated {len(sales_df)} sales transactions")
print(f" Date range: {sales_df['sale_date'].min()} → {sales_df['sale_date'].max()}")
print(f" Total revenue: ₹{sales_df['revenue'].sum():,.2f}")
sales_df.head()

 Generated 10000 sales transactions
 Date range: 2024-01-01 → 2024-12-31
 Total revenue: ₹508,315,956.70


,transaction_id,sale_date,store_id,region,product_id,product_name,category,quantity,unit_price,discount,revenue,customer_id,payment_method
0,TXN100000,2024-10-20,Bangalore-03,South,PROD1317,J.K. Rowling T-Shirt,Merchandise-T-Shirt,2,806.78,0.15,1371.53,CUST3304,Wallet
1,TXN100001,2024-12-11,Chennai-04,South,PROD1039,You can't bury them all: Poems,Books,1,3363.00,0.15,2858.55,CUST3627,Wallet
2,TXN100002,2024-04-24,Chennai-04,South,PROD1237,Inspiron 15,Electronics-Laptop,5,59679.20,0.20,238716.80,CUST2312,UPI
3,TXN100003,2024-09-08,Chennai-04,South,PROD1181,Miss Peregrine’s Home for Peculiar Children (M...,Books,5,1076.00,0.10,4842.00,CUST3257,Cash
4,TXN100004,2024-04-18,Chennai-04,South,PROD1018,Libertarianism for Beginners,Books,3,5133.00,0.10,13859.10,CUST3525,Cash


In [21]:
print("=" * 60)
print(" WEB SCRAPING PROJECT — DATA SUMMARY")
print("=" * 60)
print(f"\n Total Products Scraped : {len(products_df)}")
print(f" Number of Sources      : {products_df['source'].nunique()}")
print(f"  Unique Categories     : {products_df['category'].nunique()}")
print(f" Total Transactions     : {len(sales_df)}")
print(f" Stores Covered         : {sales_df['store_id'].nunique()}")
print(f" Unique Customers       : {sales_df['customer_id'].nunique()}")
print(f" Total Revenue          : ₹{sales_df['revenue'].sum():,.2f}")
print(f" Total Units Sold       : {sales_df['quantity'].sum():,}")

print("\n Revenue by Store:")
print(sales_df.groupby("store_id")["revenue"].sum().sort_values(ascending=False))

print("\n Revenue by Category:")
print(sales_df.groupby("category")["revenue"].sum().sort_values(ascending=False).head(10))

print("\n Files saved:")
print("   - data/products_master.csv")
print("   - data/scraped_sales.csv")

 WEB SCRAPING PROJECT — DATA SUMMARY

 Total Products Scraped : 398
 Number of Sources      : 3
  Unique Categories     : 7
 Total Transactions     : 10000
 Stores Covered         : 5
 Unique Customers       : 1984
 Total Revenue          : ₹507,520,454.74
 Total Units Sold       : 29,928

 Revenue by Store:
store_id
Mumbai-01       1.112443e+08
Hyderabad-05    1.008964e+08
Bangalore-03    9.927720e+07
Delhi-02        9.906043e+07
Chennai-04      9.704220e+07
Name: revenue, dtype: float64

 Revenue by Category:
category
Electronics-Laptop     4.224746e+08
Books                  4.808879e+07
Electronics-Tablet     2.178271e+07
Electronics-Phone      1.091828e+07
Merchandise-T-Shirt    1.519245e+06
Merchandise-Mug        1.431928e+06
Merchandise-Poster     1.304855e+06
Name: revenue, dtype: float64

 Files saved:
   - data/products_master.csv
   - data/scraped_sales.csv


In [10]:
import urllib.parse
from sqlalchemy import create_engine, text
import pandas as pd

# 1. Raw Configuration Parameters
DB_USER = "root"
DB_PASS = "Gourav@--"  # Your password containing '@'
DB_HOST = "localhost"
DB_PORT = "3306"
DB_NAME = "sales_dashboard_db"

# 2. URL-encode the password safely to handle special characters like '@'
safe_password = urllib.parse.quote_plus(DB_PASS)

# 3. Establish Connectivity via PyMySQL using the safe password string
db_url = f"mysql+pymysql://{DB_USER}:{safe_password}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(db_url)

print(" Pushing DataFrames to MySQL Environment...")

# 4. Write DataFrames to SQL Tables
products_df.to_sql(name="products", con=engine, if_exists="replace", index=False)
sales_df.to_sql(name="sales", con=engine, if_exists="replace", index=False)

print(" Successfully migrated both 'products' and 'sales' tables to MySQL!")

 Pushing DataFrames to MySQL Environment...
 Successfully migrated both 'products' and 'sales' tables to MySQL!
